# Convert the data from db to parquet file format

In [ ]:
import duckdb
from pathlib import Path
import time

# ----------------------------
# CONFIG
# ----------------------------
db_file = Path('../data/landmark_database.db').resolve()  # path to your SQLite DB
table_name = 'landmarks'                                   # table you want to export
parquet_file = Path('../data/landmarks.parquet').resolve()        # output Parquet file
columns_to_export = 'id, patient_name, frame, movement_type, side, timestamp_ms, landmark_id, x_norm, y_norm, z_norm, visibility, x_px, y_px, dataset, gait_pattern, add_pattern_info, title, fps, width, height, gait_markers'                                    # list columns, or '*' for all
compression = 'ZSTD'                                       # 'SNAPPY' is faster but bigger, 'ZSTD' compresses better

# ----------------------------
# CONNECT TO DUCKDB
# ----------------------------
con = duckdb.connect()
con.execute("INSTALL sqlite; LOAD sqlite;")

# ----------------------------
# GET ROW COUNT (for progress estimate)
# ----------------------------
count_query = f"""
SELECT COUNT(*) AS total_rows
FROM sqlite_scan('{db_file}', '{table_name}')
"""
total_rows = con.execute(count_query).fetchone()[0]
print(f"Total rows to export: {total_rows:,}")

# ----------------------------
# EXPORT TO PARQUET
# ----------------------------
start_time = time.time()

export_query = f"""
COPY (
    SELECT {columns_to_export}
    FROM sqlite_scan('{db_file}', '{table_name}')
)
TO '{parquet_file}'
(FORMAT PARQUET, COMPRESSION {compression});
"""

print("Starting export to Parquet...")
con.execute(export_query)

end_time = time.time()
elapsed = end_time - start_time
print(f"Export finished in {elapsed/60:.2f} minutes")
print(f"Parquet saved to: {parquet_file}")


Total rows to export: 21,906,020
Starting export to Parquet...
Export finished in 0.41 minutes
Parquet saved to: /Users/marcbp/spiced_bootcamp/Capstone Project/GAITy-Capstone-Modeling/notebooks/landmarks.parquet


### Compare the row and columns from the parquet and .db database file

In [ ]:
# check parquet file
import polars as pl

# Load lazily
lf_parquet = pl.scan_parquet("../data/landmarks.parquet")

# 1️⃣ Number of rows
parquet_rows = lf_parquet.select(pl.len()).collect()[0, 0]

# 2️⃣ Number of columns
parquet_columns = len(lf_parquet.columns)

# 3️⃣ Column names
parquet_colnames = lf_parquet.columns

print(f"Parquet rows: {parquet_rows}")
print(f"Parquet columns: {parquet_columns}")
print(f"Column names: {parquet_colnames}")


Parquet rows: 21906020
Parquet columns: 21
Column names: ['id', 'patient_name', 'frame', 'movement_type', 'side', 'timestamp_ms', 'landmark_id', 'x_norm', 'y_norm', 'z_norm', 'visibility', 'x_px', 'y_px', 'dataset', 'gait_pattern', 'add_pattern_info', 'title', 'fps', 'width', 'height', 'gait_markers']


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2359/3560263793.py:10: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  parquet_columns = len(lf_parquet.columns)
/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2359/3560263793.py:13: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  parquet_colnames = lf_parquet.columns


In [11]:
#check .db vile
import sqlite3

# Connect to your SQLite DB
conn = sqlite3.connect("../data/landmark_database.db")
cur = conn.cursor()

# 1️⃣ Pick your table name
table_name = "landmarks"

# 2️⃣ Number of rows
cur.execute(f"SELECT COUNT(*) FROM {table_name}")
db_rows = cur.fetchone()[0]

# 3️⃣ Column names
cur.execute(f"PRAGMA table_info({table_name})")
db_columns = [col[1] for col in cur.fetchall()]
db_column_count = len(db_columns)

print(f"SQLite rows: {db_rows}")
print(f"SQLite columns: {db_column_count}")
print(f"Column names: {db_columns}")

conn.close()


SQLite rows: 21906020
SQLite columns: 36
Column names: ['id', 'patient_name', 'frame', 'movement_type', 'jacket_status', 'side', 'model_name', 'timestamp_ms', 'landmark_id', 'x_norm', 'y_norm', 'z_norm', 'visibility', 'x_px', 'y_px', 'source_file', 'file_order', 'file_path', 'start_frame', 'end_frame', 'url', 'gait_event', 'dataset', 'gait_pattern', 'add_pattern_info', 'title', 'uploader', 'fps', 'start_time', 'end_time', 'duration', 'checksum', 'width', 'height', 'gait_markers', 'created_at']


In [12]:
# get results of size check
print(f"Row difference: {db_rows - parquet_rows}")
print(f"Column difference: {db_column_count - parquet_columns}")
missing_cols = set(db_columns) - set(parquet_colnames)
print(f"Columns in DB but missing in Parquet: {missing_cols}")


Row difference: 0
Column difference: 15
Columns in DB but missing in Parquet: {'gait_event', 'source_file', 'url', 'start_time', 'end_frame', 'model_name', 'start_frame', 'end_time', 'created_at', 'file_path', 'jacket_status', 'duration', 'checksum', 'uploader', 'file_order'}


# Load the data lazyly

In [47]:
import polars as pl

lf = pl.scan_parquet("../data/landmarks.parquet")


### Filter the rows out that contain empty landmarkers a the beginning and end of the time series of the snippet

In [48]:
# 2️⃣ Mark invalid rows
lf = lf.with_columns(
    (
        pl.col("x_norm").is_null() |
        pl.col("y_norm").is_null() |
        pl.col("z_norm").is_null()
    ).alias("row_invalid")
)

#2️⃣ 🔴 NEW: promote row invalidity → frame invalidity
lf = lf.with_columns(
    pl.any("row_invalid")
      .over(["patient_name", "frame"])
      .alias("frame_invalid")
)


# 3️⃣ Sort
lf = lf.sort(["patient_name", "frame"])

# 4️⃣ Detect sequences
lf = lf.with_columns(
    (
        (pl.col("frame") - pl.col("frame").shift(1).over("patient_name") != 1)
        .fill_null(True)
        .cast(pl.UInt32)
    )
    .cum_sum()
    .over("patient_name")
    .alias("sequence_id")
)


# 5️⃣ Assign a row index per (patient, sequence)
lf = lf.with_columns(
    pl.int_range(0, pl.len())
      .over(["patient_name", "sequence_id"])
      .alias("row_idx")
)



# 6️⃣ Find the first and last valid frame row per sequence
bounds = (
    lf.filter(~pl.col("frame_invalid"))
      .group_by(["patient_name", "sequence_id"])
      .agg([
          pl.min("row_idx").alias("first_valid_idx"),
          pl.max("row_idx").alias("last_valid_idx"),
      ])
)

# Trim only boundary invalid frames 
lf_clean = (
    lf.join(bounds, on=["patient_name", "sequence_id"], how="inner")
      .filter(
          (pl.col("row_idx") >= pl.col("first_valid_idx")) &
          (pl.col("row_idx") <= pl.col("last_valid_idx"))
      )
      .drop([
          "row_invalid",
          "frame_invalid",
          "row_idx",
          "first_valid_idx",
          "last_valid_idx",
          "sequence_id",
      ])
)

# Save the cleaned dataset
lf_clean.sink_parquet("../data/landmarks_cleaned.parquet")


#Final sanity check (this should be empty)
check = (
    lf_clean
    .with_columns(
        (
            pl.col("x_norm").is_null() |
            pl.col("y_norm").is_null() |
            pl.col("z_norm").is_null()
        ).alias("row_invalid")
    )
    .group_by(["patient_name", "frame"])
    .agg(
        pl.any("row_invalid").alias("frame_invalid")
    )
    .group_by("patient_name")
    .agg([
        pl.first("frame_invalid").alias("first_frame_invalid"),
        pl.last("frame_invalid").alias("last_frame_invalid"),
    ])
    .collect()
)

print(check.filter(pl.col("first_frame_invalid") | pl.col("last_frame_invalid")))


shape: (0, 3)
┌──────────────┬─────────────────────┬────────────────────┐
│ patient_name ┆ first_frame_invalid ┆ last_frame_invalid │
│ ---          ┆ ---                 ┆ ---                │
│ str          ┆ bool                ┆ bool               │
╞══════════════╪═════════════════════╪════════════════════╡
└──────────────┴─────────────────────┴────────────────────┘


In [49]:
# Confirm no boundary nulls remain (hard assertion)
assert lf_clean.select(
    (
        (pl.col("x_norm").is_not_null() &
         pl.col("y_norm").is_not_null() &
         pl.col("z_norm").is_not_null()).first() &
        (pl.col("x_norm").is_not_null() &
         pl.col("y_norm").is_not_null() &
         pl.col("z_norm").is_not_null()).last()
    )
).collect().item(), "Boundary nulls detected!"


In [39]:
# save the cleaned dataset
lf_clean.sink_parquet("../data/landmarks_cleaned.parquet")

In [52]:
# Spot-check one patient visually
pl.read_parquet("../data/landmarks_cleaned.parquet") \
  .filter(pl.col("patient_name") == "PA002") \
  .select(["frame", "x_norm", "y_norm", "z_norm"]) \
  .head(40)


frame,x_norm,y_norm,z_norm
i64,f64,f64,f64
38,0.221827,0.357915,-0.092667
38,0.243351,0.393111,0.063103
38,0.281515,0.490929,0.060424
38,0.799126,0.510871,-0.037107
38,0.834745,0.407479,0.028849
…,…,…,…
39,0.893708,0.161204,-0.109369
39,0.890391,0.160846,-0.09039
39,0.830499,0.407398,0.029333


In [53]:
# check cleaned parquet file

pl.scan_parquet("../data/landmarks_cleaned.parquet") \
  .select(pl.len()) \
  .collect()



len
u32
18641130


This should now contain the full frames after removing frames at the beginning and end of the sequence for each patient that had null values inside the coordinates x_norm, y_norm and z_norm.

## Check for null values for the coordinates and check if they are consecutive

In [55]:


# 1️⃣ Load cleaned dataset
lf_clean = pl.scan_parquet("../data/landmarks_cleaned.parquet")

# 2️⃣ Mark invalid coordinate rows
lf_clean = lf_clean.with_columns(
    (
        pl.col("x_norm").is_null() |
        pl.col("y_norm").is_null() |
        pl.col("z_norm").is_null()
    ).alias("invalid_flag")
)

# 3️⃣ Sort by patient + landmark + frame
lf_clean = lf_clean.sort(["patient_name", "landmark_id", "frame"])

# 4️⃣ Compute frame difference from previous row per patient + landmark
lf_clean = lf_clean.with_columns(
    (pl.col("frame") - pl.col("frame").shift(1).over(["patient_name","landmark_id"])).alias("frame_diff"),
)

# 5️⃣ Identify start of new invalid sequences
lf_clean = lf_clean.with_columns(
    (
        (pl.col("invalid_flag") & 
         (
            (pl.col("frame_diff") != 1) | 
            (~pl.col("invalid_flag").shift(1).over(["patient_name","landmark_id"]).fill_null(False))
         )
        )
        .cast(pl.UInt32)  # 1 = new sequence, 0 = continuation
    ).alias("new_invalid_sequence")
)

# 6️⃣ Assign a unique ID to each consecutive invalid sequence
# cumulative sum of new_invalid_sequence per patient + landmark
lf_clean = lf_clean.with_columns(
    pl.sum("new_invalid_sequence").over(["patient_name","landmark_id"]).alias("invalid_sequence_id")
)

# 7️⃣ Aggregate info per invalid sequence
invalid_sequences = (
    lf_clean
    .filter(pl.col("invalid_flag"))  # only invalid rows
    .group_by(["patient_name","landmark_id","invalid_sequence_id"])
    .agg([
        pl.min("frame").alias("start_frame"),
        pl.max("frame").alias("end_frame"),
        pl.count().alias("num_frames")
    ])
    .sort(["patient_name","landmark_id","start_frame"])
    .collect()
)

print(invalid_sequences)


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2359/1584208013.py:48: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_frames")


shape: (21, 6)
┌───────────────────────┬─────────────┬─────────────────────┬─────────────┬───────────┬────────────┐
│ patient_name          ┆ landmark_id ┆ invalid_sequence_id ┆ start_frame ┆ end_frame ┆ num_frames │
│ ---                   ┆ ---         ┆ ---                 ┆ ---         ┆ ---       ┆ ---        │
│ str                   ┆ i64         ┆ u32                 ┆ i64         ┆ i64       ┆ u32        │
╞═══════════════════════╪═════════════╪═════════════════════╪═════════════╪═══════════╪════════════╡
│ cljw4bg4q002e3n6lkugx ┆ 0           ┆ 1                   ┆ 900         ┆ 904       ┆ 5          │
│ 2mrd                  ┆             ┆                     ┆             ┆           ┆            │
│ cljw70yj4003d3n6lzhjn ┆ 0           ┆ 1                   ┆ 568         ┆ 579       ┆ 12         │
│ 55nb                  ┆             ┆                     ┆             ┆           ┆            │
│ cljw743d9003q3n6l8tpm ┆ 0           ┆ 1                   ┆ 124         ┆ 

In [56]:
# Count how many patients have at least one invalid sequence
num_patients_with_invalids = (
    invalid_sequences
    .select(pl.col("patient_name").unique())
    .height
)

print(f"Number of patients with at least one invalid sequence: {num_patients_with_invalids}")


Number of patients with at least one invalid sequence: 21


In [57]:
# Filter sequences where invalid spans more than 1 frame
long_invalids = invalid_sequences.filter(pl.col("num_frames") > 1)

# Count unique patients affected
num_patients_long_invalids = long_invalids.select(pl.col("patient_name").unique()).height

print(f"Number of patients with consecutive null frames > 1: {num_patients_long_invalids}")


Number of patients with consecutive null frames > 1: 20


In [60]:
# If `invalid_sequences` is already an eager DataFrame
# Filter sequences where invalid spans more than 1 frame
long_invalids = invalid_sequences.filter(pl.col("num_frames") > 1)

# Sort for readability
long_invalids = long_invalids.sort(["patient_name", "landmark_id", "start_frame"])

# Show the full table directly
print(long_invalids)

shape: (20, 6)
┌───────────────────────┬─────────────┬─────────────────────┬─────────────┬───────────┬────────────┐
│ patient_name          ┆ landmark_id ┆ invalid_sequence_id ┆ start_frame ┆ end_frame ┆ num_frames │
│ ---                   ┆ ---         ┆ ---                 ┆ ---         ┆ ---       ┆ ---        │
│ str                   ┆ i64         ┆ u32                 ┆ i64         ┆ i64       ┆ u32        │
╞═══════════════════════╪═════════════╪═════════════════════╪═════════════╪═══════════╪════════════╡
│ cljw4bg4q002e3n6lkugx ┆ 0           ┆ 1                   ┆ 900         ┆ 904       ┆ 5          │
│ 2mrd                  ┆             ┆                     ┆             ┆           ┆            │
│ cljw70yj4003d3n6lzhjn ┆ 0           ┆ 1                   ┆ 568         ┆ 579       ┆ 12         │
│ 55nb                  ┆             ┆                     ┆             ┆           ┆            │
│ cljxkx17i000c3n6ldi91 ┆ 0           ┆ 1                   ┆ 527         ┆ 

Export list of patients with null values in their coordinates to a csv file

In [61]:
# Load cleaned dataset (if not already)
lf_clean = pl.read_parquet("../data/landmarks_cleaned.parquet")  # eager DataFrame

# Mark rows with any null coordinates
lf_clean = lf_clean.with_columns(
    (
        pl.col("x_norm").is_null() |
        pl.col("y_norm").is_null() |
        pl.col("z_norm").is_null()
    ).alias("row_invalid")
)

# Select all unique patients who have at least one invalid row
patients_with_invalids = (
    lf_clean
    .filter(pl.col("row_invalid"))
    .select(pl.col("patient_name").unique())
    .sort("patient_name")
)

# Export to CSV
patients_with_invalids.write_csv("../data/patients_with_invalid_coordinates.csv")

print(f"Exported {patients_with_invalids.height} patients to CSV.")















Exported 21 patients to CSV.


In [62]:
# Assuming `invalid_sequences` is your table with consecutive invalid sequences
# Columns: patient_name, landmark_id, invalid_sequence_id, start_frame, end_frame, num_frames

# Filter sequences with at least 1 invalid frame
patients_invalid_full = invalid_sequences.filter(pl.col("num_frames") > 0)

# Sort for readability
patients_invalid_full = patients_invalid_full.sort(["patient_name", "landmark_id", "start_frame"])

# Export to CSV
patients_invalid_full.write_csv("../data/patients_invalid_sequences.csv")

print(f"Exported {patients_invalid_full.height} sequences to CSV.")

Exported 21 sequences to CSV.


check if the missing coordinates are the same or different

In [86]:


# 1️⃣ Read cleaned dataset
lf_clean = pl.read_parquet("../data/landmarks_cleaned.parquet")

# 2️⃣ Keep only patients that had nulls before
patients_with_invalid = long_invalids["patient_name"].to_list()
lf_invalid = lf_clean.filter(pl.col("patient_name").is_in(patients_with_invalid))

# 3️⃣ Mark nulls per coordinate
lf_invalid = lf_invalid.with_columns([
    pl.col("x_norm").is_null().alias("x_null"),
    pl.col("y_norm").is_null().alias("y_null"),
    pl.col("z_norm").is_null().alias("z_null"),
    (pl.col("x_norm").is_null() | pl.col("y_norm").is_null() | pl.col("z_norm").is_null()).alias("row_invalid")
])

# 4️⃣ Assign a row index per patient + landmark to track consecutive frames
lf_invalid = lf_invalid.sort(["patient_name", "landmark_id", "frame"])
lf_invalid = lf_invalid.with_columns(
    pl.int_range(0, pl.count()).over(["patient_name","landmark_id"]).alias("row_idx")
)

# 5️⃣ Detect consecutive invalid sequences per patient + landmark
lf_invalid = lf_invalid.with_columns([
    (
        pl.col("row_invalid") & 
        (~pl.col("row_invalid").shift(1).over(["patient_name","landmark_id"])).fill_null(True)
    ).alias("new_seq_flag")
])

# 6️⃣ Assign sequence IDs by forward filling an integer marker
# We use `pl.when` + `pl.int_range` + forward_fill to create unique IDs without cumsum
new_seq_rows = lf_invalid.filter(pl.col("new_seq_flag")).select([
    "patient_name", "landmark_id", "row_idx"
]).with_columns([
    pl.int_range(1, pl.count() + 1).alias("invalid_sequence_id")
])

lf_invalid = lf_invalid.join(new_seq_rows, on=["patient_name","landmark_id","row_idx"], how="left")
lf_invalid = lf_invalid.with_columns([
    pl.col("invalid_sequence_id").forward_fill().alias("invalid_sequence_id")
])

# 7️⃣ Aggregate per invalid sequence to count frames per coordinate
invalid_coords_summary = (
    lf_invalid
    .filter(pl.col("row_invalid"))
    .group_by(["patient_name","landmark_id","invalid_sequence_id"])
    .agg([
        pl.sum("x_null").alias("num_x_null"),
        pl.sum("y_null").alias("num_y_null"),
        pl.sum("z_null").alias("num_z_null"),
        pl.count("row_invalid").alias("num_frames"),
        pl.min("frame").alias("start_frame"),
        pl.max("frame").alias("end_frame"),
    ])
    .sort(["patient_name","landmark_id","invalid_sequence_id"])
)

# 8️⃣ Export to CSV
invalid_coords_summary.write_csv("../data/coordinates_invalid_sequences.csv")
print(f"Exported {invalid_coords_summary.height} invalid sequences with per-coordinate counts.")


Exported 23 invalid sequences with per-coordinate counts.


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2359/1791204823.py:19: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.int_range(0, pl.count()).over(["patient_name","landmark_id"]).alias("row_idx")
/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2359/1791204823.py:35: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.int_range(1, pl.count() + 1).alias("invalid_sequence_id")


## Filter for the joints we want

In [87]:

# 1️⃣ Load the cleaned dataset
lf_clean = pl.read_parquet("../data/landmarks_cleaned.parquet")

# 2️⃣ Define the landmarks you want to keep
GAIT_JOINTS = [
    2, 5,     # eyes (head orientation)
    11, 12,   # shoulders
    23, 24,   # hips
    25, 26,   # knees
    27, 28,   # ankles
    29, 30,   # heels
    31, 32    # foot index
]

# 3️⃣ Filter the dataset to keep only those landmarks
lf_gait = lf_clean.filter(pl.col("landmark_id").is_in(GAIT_JOINTS))

# 4️⃣ Save as a new parquet
lf_gait.write_parquet("../data/landmarks_gait_joints.parquet")

print(f"Saved {lf_gait.height} rows for the selected gait landmarks.")


Saved 7908306 rows for the selected gait landmarks.


In [ ]:
# check if there are still null values for the coordinates

# Load the filtered gait joints dataset
lf_gait = pl.read_parquet("../data/landmarks_gait_joints.parquet")

# Check for any nulls per coordinate
null_summary = lf_gait.select([
    pl.col("x_norm").is_null().sum().alias("num_x_null"),
    pl.col("y_norm").is_null().sum().alias("num_y_null"),
    pl.col("z_norm").is_null().sum().alias("num_z_null"),
    pl.count().alias("total_rows")
])

print(null_summary)


shape: (1, 4)
┌────────────┬────────────┬────────────┬────────────┐
│ num_x_null ┆ num_y_null ┆ num_z_null ┆ total_rows │
│ ---        ┆ ---        ┆ ---        ┆ ---        │
│ u32        ┆ u32        ┆ u32        ┆ u32        │
╞════════════╪════════════╪════════════╪════════════╡
│ 0          ┆ 0          ┆ 0          ┆ 7908306    │
└────────────┴────────────┴────────────┴────────────┘


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_2359/569863525.py:11: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_rows")


In [111]:
lf_gait.head()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers
i64,str,i64,str,str,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,i64,i64,str
800401,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,31,0.472943,0.647974,-0.011157,0.889764,454.0,349.0,null,null,null,null,null,null,null,null
800395,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,25,0.473669,0.482561,-0.03965,0.883199,454.0,260.0,null,null,null,null,null,null,null,null
800375,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,5,0.48507,0.100134,-0.069409,0.99501,465.0,54.0,null,null,null,null,null,null,null,null
786607,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,25,0.473078,0.48078,-0.043849,0.953473,454.0,259.0,null,null,null,null,null,null,null,null
793483,"""PA019""",144,"""Regular Movement""","""Left""",4800.0,24,0.624343,0.361788,-0.065268,0.999978,599.0,195.0,null,null,null,null,null,null,null,null


In [112]:
# Get unique values per column
unique_values = {
    "height": lf_gait.select(pl.col("height").unique()).to_series().to_list(),
    "width": lf_gait.select(pl.col("width").unique()).to_series().to_list(),
    "fps": lf_gait.select(pl.col("fps").unique()).to_series().to_list(),
}

print(unique_values)

{'height': [None, 720, 1080], 'width': [None, 1280, 1920], 'fps': [None, 60.0]}


In [113]:
# Fill null values in the columns height, width, fps
# Fill nulls with the given values
lf_filled = lf_gait.with_columns([
    pl.col("height").fill_null(540),
    pl.col("width").fill_null(960),
    pl.col("fps").fill_null(33),
    pl.col("dataset").fill_null('normal'),
    pl.col("gait_pattern").fill_null('normal'),
    pl.col("add_pattern_info").fill_null('normal'),
    pl.col("title").fill_null('normal')
])

In [114]:
lf_filled.head()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers
i64,str,i64,str,str,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,i64,i64,str
800401,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,31,0.472943,0.647974,-0.011157,0.889764,454.0,349.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null
800395,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,25,0.473669,0.482561,-0.03965,0.883199,454.0,260.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null
800375,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,5,0.48507,0.100134,-0.069409,0.99501,465.0,54.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null
786607,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,25,0.473078,0.48078,-0.043849,0.953473,454.0,259.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null
793483,"""PA019""",144,"""Regular Movement""","""Left""",4800.0,24,0.624343,0.361788,-0.065268,0.999978,599.0,195.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null


## Mapping of gait anomaly to marker

In [121]:


# 2️⃣ CLASS_MAP
CLASS_MAP = {
    "gait_anomaly_distal_foot_control_deficit": {
        "Foot Drop", "Foot Slap", "Inadequate Dorsiflexion",
        "Foot Flat Initial Contact", "Excess Pronation", "Excess Supination",
        "Reduced Metatarsophalangeal Joint Extension",
        "Absent Heel Rise During Terminal Stance", "Early Heel Rise", "Steppage Gait",
    },
    "gait_anomaly_knee_sagittal_plane_abnormality": {
        "Knee Extensor Thrust", "Knee Hyperextension", "Reduced Knee Extension",
        "Reduced Knee Flexion", "Knee Valgus",
    },
    "gait_anomaly_hip_pelvic_control_deficit": {
        "Trendelenburg", "Hip Hiking", "Posterior Pelvic Tilt", "Anterior Pelvic Tilt",
        "Reduced Pelvic Rotation", "Reduced Hip Extension", "Reduced Hip Internal Rotation",
        "Circumduction", "Medial Whip",
    },
    "gait_anomaly_trunk_balance_abnormality": {
        "Reduced Arm Swing", "Forward Lean", "Left Lean", "Right Lean",
        "Reduced Trunk Rotation", "Imbalance", "Cautious Gait",
    },
    "gait_anomaly_spatiotemporal_asymmetry": {
        "Wide Base of Support", "Step Length Asymmetry", "Reduced Step Length", "Reduced Left Weightshift",
    },
}

# 3️⃣ Flatten keyword -> class mapping
keyword_to_class = {}
for anomaly_class, keywords in CLASS_MAP.items():
    for kw in keywords:
        keyword_to_class[kw.lower()] = anomaly_class

# 4️⃣ Build the gait_anomaly expression
expr = None
for kw, anomaly_class in keyword_to_class.items():
    cond = pl.col("gait_markers").str.to_lowercase().str.contains(kw)
    if expr is None:
        expr = pl.when(cond).then(pl.lit(anomaly_class))
    else:
        expr = expr.when(cond).then(pl.lit(anomaly_class))

expr = expr.otherwise(None).alias("gait_anomaly")

# 5️⃣ Add the column

# ✅ If using DataFrame directly
lf_filled = lf_filled.with_columns(expr)

# ✅ If using LazyFrame
# lf_filled = lf_filled.with_columns(expr)
# df_filled = lf_filled.collect()  # convert LazyFrame to DataFrame

# 6️⃣ Filter non-null gait anomalies
non_null_anomalies = lf_filled.filter(pl.col("gait_anomaly").is_not_null())

# 7️⃣ View first 20 rows
print(non_null_anomalies.head(20))


shape: (20, 22)
┌──────────┬──────────────┬───────┬──────────────┬───┬───────┬────────┬──────────────┬─────────────┐
│ id       ┆ patient_name ┆ frame ┆ movement_typ ┆ … ┆ width ┆ height ┆ gait_markers ┆ gait_anomal │
│ ---      ┆ ---          ┆ ---   ┆ e            ┆   ┆ ---   ┆ ---    ┆ ---          ┆ y           │
│ i64      ┆ str          ┆ i64   ┆ ---          ┆   ┆ i64   ┆ i64    ┆ str          ┆ ---         │
│          ┆              ┆       ┆ str          ┆   ┆       ┆        ┆              ┆ str         │
╞══════════╪══════════════╪═══════╪══════════════╪═══╪═══════╪════════╪══════════════╪═════════════╡
│ 18221863 ┆ cljary1c800e ┆ 381   ┆ null         ┆ … ┆ 1920  ┆ 1080   ┆ Foot Drop    ┆ gait_anomal │
│          ┆ y3n6lak8hjn7 ┆       ┆              ┆   ┆       ┆        ┆ Foot Flat    ┆ y_distal_fo │
│          ┆ d            ┆       ┆              ┆   ┆       ┆        ┆ Initial Co…  ┆ ot_contr…   │
│ 18221881 ┆ cljary1c800e ┆ 381   ┆ null         ┆ … ┆ 1920  ┆ 1080   ┆ Foo

In [122]:
lf_filled.head()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,gait_anomaly
i64,str,i64,str,str,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,i64,i64,str,str
800401,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,31,0.472943,0.647974,-0.011157,0.889764,454.0,349.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,null
800395,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,25,0.473669,0.482561,-0.03965,0.883199,454.0,260.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,null
800375,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,5,0.48507,0.100134,-0.069409,0.99501,465.0,54.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,null
786607,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,25,0.473078,0.48078,-0.043849,0.953473,454.0,259.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,null
793483,"""PA019""",144,"""Regular Movement""","""Left""",4800.0,24,0.624343,0.361788,-0.065268,0.999978,599.0,195.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,null,null


In [123]:
# check if imputing worked
non_null_anomalies = lf_filled.filter(
    pl.col("gait_anomaly").is_not_null()
).select(["patient_name", "gait_markers", "gait_anomaly"])

print(non_null_anomalies)



shape: (849_408, 3)
┌───────────────────────────┬─────────────────────────────────┬─────────────────────────────────┐
│ patient_name              ┆ gait_markers                    ┆ gait_anomaly                    │
│ ---                       ┆ ---                             ┆ ---                             │
│ str                       ┆ str                             ┆ str                             │
╞═══════════════════════════╪═════════════════════════════════╪═════════════════════════════════╡
│ cljary1c800ey3n6lak8hjn7d ┆ Foot Drop Foot Flat Initial Co… ┆ gait_anomaly_distal_foot_contr… │
│ cljary1c800ey3n6lak8hjn7d ┆ Foot Drop Foot Flat Initial Co… ┆ gait_anomaly_distal_foot_contr… │
│ cljary1c800ey3n6lak8hjn7d ┆ Foot Drop Foot Flat Initial Co… ┆ gait_anomaly_distal_foot_contr… │
│ cljary1c800ey3n6lak8hjn7d ┆ Foot Drop Foot Flat Initial Co… ┆ gait_anomaly_distal_foot_contr… │
│ cljary1c800ey3n6lak8hjn7d ┆ Foot Drop Foot Flat Initial Co… ┆ gait_anomaly_distal_foot_contr… │


In [127]:
# fill null values for gait markers and gait_anomaly
lf_filled = lf_filled.with_columns([
    pl.col("gait_anomaly").fill_null("normal"),
    pl.col("gait_markers").fill_null("normal"),
    pl.col("movement_type").fill_null("abnormal")
])

In [125]:
lf_filled.head()

id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,visibility,x_px,y_px,dataset,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,gait_anomaly
i64,str,i64,str,str,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,i64,i64,str,str
800401,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,31,0.472943,0.647974,-0.011157,0.889764,454.0,349.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,"""normal""","""normal"""
800395,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,25,0.473669,0.482561,-0.03965,0.883199,454.0,260.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,"""normal""","""normal"""
800375,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,5,0.48507,0.100134,-0.069409,0.99501,465.0,54.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,"""normal""","""normal"""
786607,"""PA019""",143,"""Regular Movement""","""Right""",4766.0,25,0.473078,0.48078,-0.043849,0.953473,454.0,259.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,"""normal""","""normal"""
793483,"""PA019""",144,"""Regular Movement""","""Left""",4800.0,24,0.624343,0.361788,-0.065268,0.999978,599.0,195.0,"""normal""","""normal""","""normal""","""normal""",33.0,960,540,"""normal""","""normal"""


In [128]:
# Check for remaining nulls in all columns
print(lf_filled.null_count())


shape: (1, 22)
┌─────┬──────────────┬───────┬───────────────┬───┬───────┬────────┬──────────────┬──────────────┐
│ id  ┆ patient_name ┆ frame ┆ movement_type ┆ … ┆ width ┆ height ┆ gait_markers ┆ gait_anomaly │
│ --- ┆ ---          ┆ ---   ┆ ---           ┆   ┆ ---   ┆ ---    ┆ ---          ┆ ---          │
│ u32 ┆ u32          ┆ u32   ┆ u32           ┆   ┆ u32   ┆ u32    ┆ u32          ┆ u32          │
╞═════╪══════════════╪═══════╪═══════════════╪═══╪═══════╪════════╪══════════════╪══════════════╡
│ 0   ┆ 0            ┆ 0     ┆ 0             ┆ … ┆ 0     ┆ 0      ┆ 0            ┆ 0            │
└─────┴──────────────┴───────┴───────────────┴───┴───────┴────────┴──────────────┴──────────────┘


In [129]:
# Save to Parquet
lf_filled.write_parquet("../data/filled_gait_data.parquet")



# Steps for data cleaning

1. removed frames with leading or trailing null values for coordinates
2. check of null values for coordinates shows that only landmarker id =0 is missing once in while -> ignore because this is going to be excluded anyways.
3. only kept the landmarkers of interest
4. imputed null values
5. added anomaly classes column and for null values imputed normal
6. for null values in movement type added abnormal